In [1]:
# ============================================================
# Configuración general del notebook
# ============================================================

from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Nombre del archivo CSV de entrada.
INPUT_FILE = Path("02_loan_data.csv")

# Nombre del archivo CSV que se generará después del preprocesamiento.
OUTPUT_FILE = Path("loan_preprocesado.csv")

# Nombre de la variable objetivo.
TARGET_COL = "loan_status"

# ------------------------------------------------------------
# Parámetro principal de codificación
# ------------------------------------------------------------
# "onehot" -> crea columnas binarias para cada categoría.
# "label"  -> reemplaza cada categoría por un número entero.

METODO_CODIFICACION = "onehot"

def crear_onehot_encoder():
    """
    Crea un codificador OneHotEncoder compatible con distintas versiones de scikit-learn.

    En versiones recientes de scikit-learn se utiliza sparse_output=False.
    En versiones anteriores se utiliza sparse=False.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def cargar_dataset(ruta_archivo: Path) -> pd.DataFrame:
    """
    Lee el archivo CSV con los datos de préstamos y aplica las primeras limpiezas.

    Pasos realizados:
    1. Lee el archivo 02_loan_data.csv.
    2. Normaliza los nombres de columnas:
       - elimina espacios al inicio y al final;
       - transforma los nombres a minúsculas.
    3. Limpia las variables categóricas person_gender, person_education,
       person_home_ownership, loan_intent y previous_loan_defaults_on_file.
    4. Filtra la variable loan_status para conservar solo valores 0 y 1.
    5. Convierte loan_status a formato numérico entero.

    Retorna:
        DataFrame preprocesado en su etapa inicial.
    """
    if not ruta_archivo.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo {ruta_archivo}. "
            "Verifica que esté en la misma carpeta del notebook."
        )

    df = pd.read_csv(ruta_archivo)

    # Normalizar nombres de columnas para evitar errores por mayúsculas o espacios.
    df.columns = [c.strip().lower() for c in df.columns]

    # Limpieza de columnas categóricas específicas.
    df["person_gender"] = df["person_gender"].astype(str).str.strip()
    df["person_education"] = df["person_education"].astype(str).str.strip()
    df["person_home_ownership"] = df["person_home_ownership"].astype(str).str.strip()
    df["loan_intent"] = df["loan_intent"].astype(str).str.strip()
    df["previous_loan_defaults_on_file"] = df["previous_loan_defaults_on_file"].astype(str).str.strip()

    # La variable objetivo se limpia y se convierte a entero.
    df["loan_status"] = pd.to_numeric(df["loan_status"], errors="coerce")

    # Se conservan únicamente los registros donde loan_status sea 0 o 1.
    df = df[df["loan_status"].isin([0, 1])].copy()

    # Conversión de la variable objetivo a entero.
    df[TARGET_COL] = df[TARGET_COL].astype(int)

    return df


def separar_variables(df: pd.DataFrame, target_col: str):
    """
    Separa el dataset en variables predictoras X y variable objetivo y.

    X contiene todas las columnas excepto la variable objetivo.
    y contiene solamente la variable objetivo.
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]

    return X, y


def identificar_tipos_de_variables(X: pd.DataFrame):
    """
    Identifica automáticamente las columnas categóricas y numéricas.

    Las columnas categóricas son aquellas cuyo tipo de dato es object.
    Las columnas numéricas son todas las restantes.
    """
    categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
    numeric_features = [c for c in X.columns if c not in categorical_features]

    return categorical_features, numeric_features


def codificar_con_onehot(X: pd.DataFrame, categorical_features: list, numeric_features: list) -> pd.DataFrame:
    """
    Aplica One Hot Encoding a las variables categóricas.

    Este método crea nuevas columnas binarias para representar las categorías.
    Las variables numéricas se mantienen sin cambios mediante passthrough.
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", crear_onehot_encoder(), categorical_features),
            ("num", "passthrough", numeric_features),
        ]
    )

    X_transformado = preprocessor.fit_transform(X)

    if categorical_features:
        cat_columns = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features).tolist()
    else:
        cat_columns = []

    final_columns = cat_columns + numeric_features

    X_codificado = pd.DataFrame(
        X_transformado,
        columns=final_columns,
        index=X.index
    )

    return X_codificado


def codificar_con_labelencoder(X: pd.DataFrame, categorical_features: list) -> pd.DataFrame:
    """
    Aplica Label Encoding a las variables categóricas.

    Este método reemplaza cada categoría por un número entero.
    Por ejemplo, una columna con valores como RENT, OWN, MORTGAGE
    podría transformarse en 0, 1 y 2.

    Importante:
    Label Encoding no crea columnas nuevas. Mantiene la misma cantidad de columnas,
    pero convierte las categorías a números.
    """
    X_codificado = X.copy()

    for columna in categorical_features:
        encoder = LabelEncoder()
        X_codificado[columna] = X_codificado[columna].astype(str).str.strip()
        X_codificado[columna] = encoder.fit_transform(X_codificado[columna])

    return X_codificado


def codificar_variables(X: pd.DataFrame, metodo: str = "onehot") -> pd.DataFrame:
    """
    Codifica las variables categóricas usando el método indicado por parámetro.

    Parámetros:
        X: DataFrame con las variables predictoras.
        metodo: Tipo de codificación a utilizar.
                Valores permitidos: "onehot" o "label"

    Retorna:
        DataFrame con las variables categóricas codificadas.
    """
    metodo = metodo.lower().strip()

    categorical_features, numeric_features = identificar_tipos_de_variables(X)

    print("Variables categóricas identificadas:")
    print(categorical_features)

    print("\nVariables numéricas identificadas:")
    print(numeric_features)

    if metodo == "onehot":
        print("\nMétodo seleccionado: One Hot Encoding")
        return codificar_con_onehot(X, categorical_features, numeric_features)

    if metodo == "label":
        print("\nMétodo seleccionado: Label Encoding")
        return codificar_con_labelencoder(X, categorical_features)

    raise ValueError(
        "Método de codificación no válido. "
        "Usa 'onehot' o 'label'."
    )


def guardar_dataset_preprocesado(X_codificado: pd.DataFrame, y: pd.Series, ruta_salida: Path) -> pd.DataFrame:
    """
    Une las variables predictoras codificadas con la variable objetivo
    y guarda el resultado en un archivo CSV.

    Retorna:
        DataFrame final preprocesado.
    """
    df_preprocesado = pd.concat(
        [
            X_codificado.reset_index(drop=True),
            y.reset_index(drop=True)
        ],
        axis=1
    )

    df_preprocesado.to_csv(ruta_salida, index=False)

    return df_preprocesado


def ejecutar_preprocesamiento(metodo_codificacion: str = "onehot") -> pd.DataFrame:
    """
    Ejecuta el flujo completo de preprocesamiento.

    Flujo:
    1. Cargar 02_loan_data.csv.
    2. Limpiar columnas relevantes.
    3. Separar X e y.
    4. Codificar variables categóricas según el método elegido.
    5. Guardar loan_preprocesado.csv.

    Retorna:
        DataFrame preprocesado.
    """
    print("Iniciando preprocesamiento de datos...")

    df = cargar_dataset(INPUT_FILE)

    print(f"Dataset cargado correctamente con {len(df)} registros.")
    print(f"Columnas disponibles: {list(df.columns)}")

    X, y = separar_variables(df, TARGET_COL)

    X_codificado = codificar_variables(
        X,
        metodo=metodo_codificacion
    )

    df_preprocesado = guardar_dataset_preprocesado(
        X_codificado,
        y,
        OUTPUT_FILE
    )

    print("\nPreprocesamiento completado correctamente.")
    print(f"Archivo generado: {OUTPUT_FILE}")
    print(f"Filas del dataset final: {df_preprocesado.shape[0]}")
    print(f"Columnas del dataset final: {df_preprocesado.shape[1]}")

    return df_preprocesado


# ============================================================
# Ejecutar el preprocesamiento
# ============================================================

df_preprocesado = ejecutar_preprocesamiento(
    metodo_codificacion=METODO_CODIFICACION
)

# Visualizar las primeras filas del dataset preprocesado.
df_preprocesado.head()

Iniciando preprocesamiento de datos...
Dataset cargado correctamente con 45000 registros.
Columnas disponibles: ['person_age', 'person_gender', 'person_education', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file', 'loan_status']
Variables categóricas identificadas:
['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']

Variables numéricas identificadas:
['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']

Método seleccionado: One Hot Encoding

Preprocesamiento completado correctamente.
Archivo generado: loan_preprocesado.csv
Filas del dataset final: 45000
Columnas del dataset final: 28


,person_gender_female,person_gender_male,person_education_Associate,person_education_Bachelor,person_education_Doctorate,person_education_High School,person_education_Master,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,...,previous_loan_defaults_on_file_Yes,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,22.0,71948.0,0.0,35000.0,16.02,0.49,3.0,561.0,1
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,21.0,12282.0,0.0,1000.0,11.14,0.08,2.0,504.0,0
2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,25.0,12438.0,3.0,5500.0,12.87,0.44,3.0,635.0,1
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,23.0,79753.0,0.0,35000.0,15.23,0.44,2.0,675.0,1
4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,24.0,66135.0,1.0,35000.0,14.27,0.53,4.0,586.0,1
